# Quickstart

Model, dataset, one batch. Needs `data/openfake_train.csv` from `scripts/download_data.py`.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

import yaml, torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from src.model import Detector
from src.losses import patch_loss
from src.data import TrainDataset, load_rows

cfg = yaml.safe_load(open('configs/train.yaml'))

## Как устроена модель

**Идея.** Картинка режется на квадраты 16×16 пикселей, они называются патчами. Для каждого патча модель оценивает, сгенерирован ли он. Из этой карты получается общий балл картинки. Поэтому модель ловит и полностью сгенерированные картинки, и небольшие вставки.

**1. Готовая модель зрения.** DINOv3 ViT-L/16 от Meta уже обучена на 1.7 млрд картинок и описывает каждый патч вектором из 1024 чисел. Мы её не дообучаем. Векторы берём с трёх глубин сети, из 12-го, 18-го и 24-го блоков из 24: ранние слои видят текстуру и следы генерации, поздние видят смысл. После склейки у каждого патча 3072 числа:

$$F = [F_{12};\,F_{18};\,F_{24}] \in \mathbb{R}^{3072 \times h \times w}, \qquad h = H/16,\; w = W/16$$

**2. Обучаемая голова.** Только эта часть учится: три свёртки, около 1.4M параметров. Они превращают вектор каждого патча в одно число $z_{ij}$, то есть насколько патч похож на сгенерированный. Средняя свёртка 3×3 даёт патчу посмотреть на соседей, это окно 48×48 пикселей.

$$z = W_3 * \mathrm{GELU}\big(W_2 * \mathrm{GELU}(W_1 * F)\big) \in \mathbb{R}^{h \times w}$$

**3. Балл картинки.** Берём 5 % патчей с самыми большими $z$ и усредняем. Один максимальный патч дал бы шумный балл, а среднее по всем патчам размыло бы маленькую вставку. Для картинки 1024×1024 это 205 патчей из 4096.

$$k = \max(1, \mathrm{round}(0.05\,hw)), \qquad \ell = \frac{1}{k}\sum_{\text{top-}k} z_{ij}, \qquad s = \sigma(\ell) \in (0, 1)$$

**4. Чему учим.** Маска $M$ отмечает сгенерированные пиксели: у фейка она вся из единиц, у реального фото вся из нулей. Цель патча: доля сгенерированных пикселей в нём. Цель картинки: есть ли они вообще. Лосс складывает две бинарные кросс-энтропии.

$$t_{ij} = \operatorname{mean}_{\text{патч}(i,j)} M, \qquad y = \max M, \qquad L = \operatorname{mean}_{ij} \mathrm{BCE}(z_{ij}, t_{ij}) + \mathrm{BCE}(\ell, y)$$

В OpenFake масок нет, поэтому на обучении цель пока всегда целиком 0 или целиком 1. Локальные правки модель видит только на тесте, в SID_Set.

**5. Порог.** Порог $\tau$ подбирается только по реальным картинкам, которых модель не видела на обучении. Он ставится так, чтобы модель ложно срабатывала не больше чем на 1 % из них. Если $s > \tau$, картинка сгенерирована, а карта $\sigma(z_{ij})$ показывает, где именно.

$$\tau = s_{(\lceil (n+1)(1-\alpha) \rceil)} \text{ среди баллов } n \text{ отложенных реальных}, \qquad \alpha = 0.01$$

In [ ]:
# model: frozen DINOv3 + small head on patch features
model = Detector(**cfg['model'])
print('patch', model.patch_size, '| head params', sum(p.numel() for p in model.head.parameters()))

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

In [ ]:
# dataset: real and generated images at their own resolution, one row per image
rows = load_rows('data/openfake_train.csv')
dataset = TrainDataset(rows, cfg['train']['max_side'], model.patch_size)
batch = next(iter(DataLoader(dataset, batch_size=1, shuffle=True)))  # sizes differ: one image at a time
batch['image'].shape, batch['label']

In [ ]:
# forward and loss
patch_logits, image_logit = model(batch['image'].to(device))
loss = patch_loss(patch_logits, image_logit, batch['target'].to(device))
patch_logits.shape, image_logit, loss.item()

In [ ]:
# image, target map, predicted map
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(batch['image'][0].permute(1, 2, 0))
ax[1].imshow(batch['target'][0, 0], vmin=0, vmax=1)
ax[2].imshow(patch_logits[0, 0].sigmoid().detach().cpu(), vmin=0, vmax=1)
for a, title in zip(ax, ['image', 'target', 'prediction']):
    a.set_title(title); a.axis('off')

## Данные

**Обучение: [OpenFake](https://arxiv.org/abs/2509.09495).** Лицензия CC BY-NC 4.0, только для исследований. Реальные фото взяты из LAION и Pexels: публичные люди, новости, мемы с текстом. Для каждого фото нейросеть написала подпись, и по этим подписям картинки сгенерировали 72 модели: GPT Image, Nano Banana, Imagen, Midjourney, DALL·E 3, Flux, SD от 1.x до 3.5, файнтюны с Civitai и кадры видеомоделей. Поэтому реальные фото и фейки похожи по сюжетам, и отличить их по содержанию нельзя.

| Сплит | Строк | Объём | Что внутри |
|---|---|---|---|
| core/train | 2.3M | 3.3 TB | реальные из LAION и Pexels, 72 генератора |
| core/test | 91k | 68 GB | только новые генераторы: GPT Image 1.5 и 2, Nano Banana Pro, Midjourney 7, Flux.2 klein 9B, Z-Image Turbo и другие; реальные из DOCCI и ImageNet |

**Что берём.** Файлы train скачиваются целиком, из каждого берутся нужные картинки, и файл удаляется. В пилоте это первые 200 файлов из 608: до `n_per_model` картинок на каждый генератор из списка `generators` в конфиге и `n_real` реальных. Генераторы очень неравны: у SD 3.5 около 150k картинок, у Nano Banana 2 около двух тысяч, и у таких берём всё, что попалось. Модель `tiny-random-sana` выкидываем, её картинки сплошной шум. 15 % реальных откладываются для порога, ещё 5 % всех данных для валидации.

**Хранение и обработка.** Картинки лежат на диске как есть, байт в байт из датасета: фейки в основном PNG, реальные фото в основном JPEG. Картинки меньше 512 пикселей пропускаются. Чтобы формат не выдавал класс, на обучении каждая картинка сжимается в JPEG со случайным качеством от 50 до 95, а ещё случайно отражается, размывается и зашумляется, одинаково для обоих классов. Разрешение не меняется: картинка подаётся целиком, а сторона больше 1024 пикселей обрезается случайным кропом без уменьшения.

**Батч.** Половина батча реальные фото, а вторую половину генераторы делят поровну: $P(\text{реальное}) = \tfrac12$, $P(g) = \tfrac{1}{2G}$. Так модель ищет общее для всех генераторов, а не черты самого частого.

**Тест.** Два режима, в обоих картинка сжимается в JPEG q90, как на обучении: `native` — исходное разрешение, как модель училась; `equalized` — все картинки уменьшены и вырезаны до 512×512, проверка, что модель не опирается на размер.
- **OpenFake test:** доля пойманных фейков от невиденных генераторов при 1 % ложных срабатываний на реальных из DOCCI.
- **SID_Set:** локальные правки с масками; pixel F1 показывает, попадает ли карта патчей в маску.
- **Отрисованные тексты, интерфейсы, графики и логотипы:** сколько ложных срабатываний на нефото.

In [ ]:
# сколько картинок скачано от каждого генератора
from collections import Counter

rows = load_rows('data/openfake_train.csv')
counts = Counter(r['generator'] or 'real' for r in rows)
print(len(counts) - 1, 'генераторов,', len(rows), 'картинок')
for name, n in counts.most_common():
    print(f'{n:7d}  {name}')

In [ ]:
# 5 случайных картинок одного генератора
import random
from PIL import Image

gen = 'gpt-image-1'  # любое имя из списка выше, или 'real'
pool = [r for r in rows if (r['generator'] or 'real') == gen]
pick = random.sample(pool, min(5, len(pool)))
fig, ax = plt.subplots(1, len(pick), figsize=(4 * len(pick), 4), squeeze=False)
for a, r in zip(ax[0], pick):
    img = Image.open(r['path']).convert('RGB')
    a.imshow(img)
    a.set_title(f'{gen}\n{img.width}×{img.height}', fontsize=9)
    a.axis('off')